# 01 — デモ: Session 1 平坦スモーク（GRF可視化）

**目的:** 3層パイプラインが **動く** ことを確認。MuJoCo GUI で **緑矢印 = GRF** を見せる。

**所要:** uv workshop 環境、Quadruped-PyMPC clone 済み


## 4セッションの違い（必読）

| | **S1 本Notebook ← 今ここ** | S2 tune | S3a boxes | S3b perlin |
|---|-------------------|---------|-----------|------------|
| **scene** | **flat（平坦）** | flat | **random_boxes（箱）** | **perlin（連続起伏）** |
| **足場最適化** | **OFF** | OFF | ON | ON |
| **主な目的** | 最小構成で動作確認 | μ / 歩調チューニング | 段差・離散障害 | 連続起伏 |
| **デモGIFで見る点** | 平坦＋標準trot | 平坦＋**速いtrot** | **箱が見える** | **うねり地形** |
| **GIF** | demo_s01_flat | demo_s02_tune | demo_s03_boxes | demo_s03_perlin |

> **S1 と S2 は地形とも平坦**です。GIFの違いは **歩調（S2は step_freq=1.75 Hz の速い trot）** と **Notebook内の実験内容** です。  
> **S3a/S3b は約9秒走行**して箱・起伏地形に入るようキャプチャしています（旧GIFは短すぎて全部平坦に見えていました）。


### このセッション固有のポイント

- **地形:** `scene=flat` — チェッカー模様の平坦床のみ（障害物なし）
- **足場最適化:** OFF — デバッグを単純化
- **デモGIF:** 標準 trot（step_freq=1.4 Hz）。画面左上に `Session 1 | scene=flat` と表示
- **Notebook:** ref_z を意図的に下げて **転倒 vs 成功** をグラフで比較

![Session 1 demo](../assets/demo_s01_flat.gif)


## Step 0 — このデモで学ぶこと

| 学習項目 | 成功のサイン |
|----------|--------------|
| プリセット適用 | config.py が意図通りにパッチされる |
| 平坦trot | 30s以上転倒しない |
| GRF可視化 | 各足に緑矢印 |
| 失敗からの復帰 | ref_z 調整で立ち直れる |


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")


## Step 1 — 環境確認

In [ ]:
pympc = ROOT / "external" / "Quadruped-PyMPC"
assert pympc.is_dir(), "Run: ./scripts/setup_references.sh"
print("PyMPC OK:", pympc)
print("Preset:", ROOT / "configs/pympc_presets/session01_flat_smoke.yaml")


## Step 2 — プリセット YAML を読む（何を最小構成にしているか）

In [ ]:
preset = load_preset_yaml("session01_flat_smoke")
import yaml
print(yaml.dump(preset, allow_unicode=True))


**設計意図（MPC設計者向け）**

- `use_foothold_optimization: False` → 初回デバッグの失敗要因を排除  
- `scene: flat` → 地形変数ゼロ  
- `gait: trot` → 最安定ゲイト  
- `mu: 0.5` → 標準摩擦モデル


## Step 3 — プリセットを config.py に適用

In [ ]:
cfg_path = apply_preset("session01_flat_smoke")
print("Applied ->", cfg_path)


## Step 4 — 短時間 headless sim（4秒）で動作確認

In [ ]:
# 初回は acados codegen で数分かかることがあります
metrics = run_flat_sim(seconds=4.0)
print(f"mean_vx={metrics['mean_vx']:.3f} m/s, min_z={metrics['min_z']:.3f} m, terminated={metrics['terminated']}")


## Step 5 — ❌ 意図的失敗: ref_z が低すぎる

**症状:** 即転倒 / 胴体が沈む / 足が地面に刺さる  
**原因:** CoM目標高度が低く、MPCが十分なGRFを計画できない  
**教訓:** まず `ref_z` を疑う（Session 1 で最も多い失敗）


In [ ]:
apply_preset("session01_flat_smoke")
bad = run_flat_sim(seconds=3.0, ref_z_scale=0.85)
good = run_flat_sim(seconds=3.0, ref_z_scale=1.05)
fig = compare_runs([("FAIL ref_z×0.85", bad), ("OK ref_z×1.05", good)])
plt.show()
print("FAIL terminated:", bad["terminated"], "| OK terminated:", good["terminated"])


## Step 6 — ✅ 成功パターンの確認

- `min_z` が一定（大きく落ちない）  
- `terminated=False`  
- `mean_vx > 0`（forward 指令時）

**retry プリセット:** `session01_flat_smoke_retry`（ref_z 微増）— 本番前に試す


In [ ]:
try:
    apply_preset("session01_flat_smoke_retry")
    retry = run_flat_sim(seconds=4.0)
    print(retry)
except Exception as e:
    print("retry preset optional:", e)


## Step 7 — デモ映像（生成済み）

[`../assets/demo_s01_flat.gif`](../assets/demo_s01_flat.gif) — `capture_demo_frames.py` で生成（平坦＋標準trot）

---

## Step 8 — チェックリスト

- [ ] プリセット → sim の流れを再現できた  
- [ ] ref_z 失敗 vs 成功を **グラフで説明** できる  
- [ ] 「MPCはGRFを計画、GUI矢印で見える」と言える  
- [ ] S2/S3 との違い（地形・足場opt・目的）を説明できる  

**次:** [02_demo_session02_flat_tune.ipynb](./02_demo_session02_flat_tune.ipynb) で μ / 歩調を触る
